# GAMs con sklearn

1. Versión multivariante GAM-like en scikit-learn usando SplineTransformer y ColumnTransformer.

2. Optimización con GridSearchCV para ajustar hiperparámetros como el número de nudos y el grado del spline.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Datos simulados con 3 variables
np.random.seed(42)
n = 300
X1 = np.linspace(0, 10, n)
X2 = np.random.uniform(0, 5, n)
X3 = np.random.uniform(-3, 3, n)
y = np.sin(X1) + np.log1p(X2) - 0.5 * X3 + np.random.normal(scale=0.3, size=n)

X = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ColumnTransformer para aplicar splines a cada variable
spline_transformer = ColumnTransformer([
    ("spline_X1", SplineTransformer(n_knots=8, degree=3), ["X1"]),
    ("spline_X2", SplineTransformer(n_knots=6, degree=3), ["X2"]),
    ("spline_X3", SplineTransformer(n_knots=6, degree=3), ["X3"])
])

# Pipeline GAM-like
gam_pipeline = Pipeline([
    ("splines", spline_transformer),
    ("linear", LinearRegression())
])

# Entrenar el modelo
gam_pipeline.fit(X_train, y_train)

# Predicciones y evaluación
y_pred = gam_pipeline.predict(X_test)
print("RMSE:", root_mean_squared_error(y_test, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir parámetros para búsqueda
param_grid = {
    "splines__spline_X1__n_knots": [5, 8, 12],
    "splines__spline_X2__n_knots": [4, 6, 10],
    "splines__spline_X3__n_knots": [4, 6, 10],
    "splines__spline_X1__degree": [2, 3],
    "splines__spline_X2__degree": [2, 3],
    "splines__spline_X3__degree": [2, 3]
}

grid_search = GridSearchCV(gam_pipeline, param_grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Mejores parámetros:", grid_search.best_params_)
print("Mejor RMSE:", np.sqrt(-grid_search.best_score_))
